# **Foundations of Data Science 2025/2026**
Department of Computer Science, Faculty of Sciences, University of Porto

**Practical #6**

In this class we will get acquainted with `mlextend`, a Python module that implement (among other functions and methods) a very popular association rules algorithm: Apriori. As usual, we will go through our old friend: the iris dataset 🥰!


In [1]:
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.preprocessing import KBinsDiscretizer
from mlxtend.frequent_patterns import apriori, association_rules

In [2]:
# The function below import the iris dataset from a library of common datasets available in sklearn
iris = load_iris()
df = pd.DataFrame(iris.data, columns=iris.feature_names)
df['target'] = iris.target

In [3]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

# In order to use association rules, we need to discretize the iris dataset attributes
# We will be using the function KBinsDiscretizer from sklearn.preprocessing which allows to select the number of bins
# Using 3 bins: low/medium/high with ordinal labels
kb = KBinsDiscretizer(n_bins=3, encode='ordinal', strategy='quantile')
disc_data = kb.fit_transform(df[iris.feature_names])

In [4]:
disc_df = pd.DataFrame(disc_data, columns=iris.feature_names)
disc_df = disc_df.astype(int)
disc_df

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm)
0,0,2,0,0
1,0,1,0,0
2,0,2,0,0
3,0,1,0,0
4,0,2,0,0
...,...,...,...,...
145,2,1,2,2
146,2,0,2,2
147,2,1,2,2
148,1,2,2,2


In [6]:
# Convert numeric codes to strings (low/med/high)
mapping = {0: "low", 1: "medium", 2: "high"}
for col in disc_df.columns:
    disc_df[col] = disc_df[col].map(mapping)
disc_df

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm)
0,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN
...,...,...,...,...
145,NaN,NaN,NaN,NaN
146,NaN,NaN,NaN,NaN
147,NaN,NaN,NaN,NaN
148,NaN,NaN,NaN,NaN


In [9]:
# Now we need to prepare the dataset to mlextend apriori which only accepts booleans
one_hot = pd.get_dummies(disc_df)
one_hot

""
0
1
2
3
4
...
145
146
147
148


In [10]:
# Optionally include target labels
df["species"] = df["target"].map({0: "setosa", 1: "versicolor", 2: "virginica"})
one_hot = pd.concat([one_hot, pd.get_dummies(df["species"])], axis=1)
one_hot

,setosa,versicolor,virginica
0,True,False,False
1,True,False,False
2,True,False,False
3,True,False,False
4,True,False,False
...,...,...,...
145,False,False,True
146,False,False,True
147,False,False,True
148,False,False,True


In [11]:
# Training Apriori: obtaining frequent itemsets
frequent_itemsets = apriori(one_hot, min_support=0.1, use_colnames=True)
print("\n=== Frequent Itemsets ===")
frequent_itemsets


=== Frequent Itemsets ===


,support,itemsets
0,0.333333,(setosa)
1,0.333333,(versicolor)
2,0.333333,(virginica)


In [12]:
# Generating rules
rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.5)
print("\n=== Association Rules ===")
rules


=== Association Rules ===


,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski


In [33]:
# Show most relevant rules
# We are sorting by lift, but the rules could be sorted by support or confidence
print("\n=== Top rules sorted by lift ===")
rules.sort_values(by="lift", ascending=False).head()


=== Top rules sorted by lift ===


,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
386,"(sepal width (cm)_medium, petal length (cm)_hi...","(petal width (cm)_high, sepal length (cm)_high)",0.113333,0.253333,0.100000,0.882353,3.482972,1.0,0.071289,6.346667,0.804010,0.375000,0.842437,0.638545
434,"(versicolor, petal length (cm)_medium, sepal w...","(sepal length (cm)_medium, petal width (cm)_me...",0.166667,0.200000,0.113333,0.680000,3.400000,1.0,0.080000,2.500000,0.847059,0.447368,0.600000,0.623333
440,"(sepal length (cm)_medium, petal width (cm)_me...","(versicolor, petal length (cm)_medium, sepal w...",0.200000,0.166667,0.113333,0.566667,3.400000,1.0,0.080000,1.923077,0.882353,0.447368,0.480000,0.623333
390,"(sepal width (cm)_medium, virginica)","(petal width (cm)_high, petal length (cm)_high...",0.120000,0.246667,0.100000,0.833333,3.378378,1.0,0.070400,4.520000,0.800000,0.375000,0.778761,0.619369
384,"(sepal width (cm)_medium, petal length (cm)_hi...","(sepal length (cm)_high, virginica)",0.120000,0.246667,0.100000,0.833333,3.378378,1.0,0.070400,4.520000,0.800000,0.375000,0.778761,0.619369


## Association rules com outro dataset

* Seguimos o tutorial
  * https://rasbt.github.io/mlxtend/user_guide/frequent_patterns/association_rules/#association_rules-association-rules-generation-from-frequent-itemsets
 
  * Neste tutorial temos a parte teórica relacionado com a **Association rules**

>Dada uma regra **"A -> C"**, **A** representa o **antecedente** e **C** representa o **consequente**

| Métrica    | Fórmula simplificada | O que mede/Intrepretação| 
| -------- | :-------: | ------- |
| ```Support```  |  $Support(A \rightarrow C) = Support(A \cup C)$  | A **frequência** com que a **regra aparece** na base de dados. <br> (Range: [0, 1]) |
| ```Confidence``` | $Support(A \rightarrow C) \over Support(A)$     | A **precisão da regra**. **Quantas vezes o consequente (C) aparece quando o antecedente (A) já lá está**. <br>(Range: [0, 1]) | 
| ```Lift```  | $Confidence(A \rightarrow C) \over Support(C)$    | Se **A** e **C** são **independentes ou dependentes**. <br>\>**1: Útil** (associação positiva). <br>**1: Independentes.** <br>**<1: Associação negativa**.| 
| ```Leverage```| $Support(A \rightarrow C) - Support(A) \times Support(C)$|  A **diferença** entre a **frequência observada de A+C** e a **frequência esperada se fossem independentes**. <br> **0: Independentes**.|
| ```Conviction``` | $ 1 - Support(C) \over 1 - Confidence(A \rightarrow C)$ | O **grau de implicação da regra**. <br> Um valor alto significa que a regra raramente estaria incorreta se fossem independentes. |


### Example 1 -- Generating Association Rules from Frequent Itemsets

The generate_rules takes dataframes of frequent itemsets as produced by the apriori, fpgrowth, or fpmax functions in mlxtend.association. To demonstrate the usage of the generate_rules method, we first create a pandas DataFrame of frequent itemsets as generated by the fpgrowth function:

In [13]:
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, fpmax, fpgrowth


dataset = [['Milk', 'Onion', 'Nutmeg', 'Kidney Beans', 'Eggs', 'Yogurt'],
           ['Dill', 'Onion', 'Nutmeg', 'Kidney Beans', 'Eggs', 'Yogurt'],
           ['Milk', 'Apple', 'Kidney Beans', 'Eggs'],
           ['Milk', 'Unicorn', 'Corn', 'Kidney Beans', 'Yogurt'],
           ['Corn', 'Onion', 'Onion', 'Kidney Beans', 'Ice cream', 'Eggs']]

te = TransactionEncoder()
te_ary = te.fit(dataset).transform(dataset)
df = pd.DataFrame(te_ary, columns=te.columns_)

frequent_itemsets = fpgrowth(df, min_support=0.6, use_colnames=True)
### alternatively:
#frequent_itemsets = apriori(df, min_support=0.6, use_colnames=True)
#frequent_itemsets = fpmax(df, min_support=0.6, use_colnames=True)

frequent_itemsets

,support,itemsets
0,1.0,(Kidney Beans)
1,0.8,(Eggs)
2,0.6,(Yogurt)
3,0.6,(Onion)
4,0.6,(Milk)
5,0.8,"(Eggs, Kidney Beans)"
6,0.6,"(Yogurt, Kidney Beans)"
7,0.6,"(Eggs, Onion)"
8,0.6,"(Kidney Beans, Onion)"
9,0.6,"(Eggs, Kidney Beans, Onion)"


The **generate_rules()** function allows you to 
1. specify your metric of interest  
2. Sepecify the according threshold. 

Currently implemented measures are confidence and lift. Let's say you are interested in rules derived from the frequent itemsets only if the level of confidence is above the 70 percent threshold (min_threshold=0.7):

In [14]:
from mlxtend.frequent_patterns import association_rules

association_rules(frequent_itemsets, metric="confidence", min_threshold=0.7)

C:\Users\Jessica\AppData\Roaming\Python\Python313\site-packages\mlxtend\frequent_patterns\association_rules.py:186: RuntimeWarning: invalid value encountered in divide
  cert_metric = np.where(certainty_denom == 0, 0, certainty_num / certainty_denom)


,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
0,(Eggs),(Kidney Beans),0.8,1.0,0.8,1.00,1.00,1.0,0.00,inf,0.0,0.80,0.000,0.900
1,(Kidney Beans),(Eggs),1.0,0.8,0.8,0.80,1.00,1.0,0.00,1.0,0.0,0.80,0.000,0.900
2,(Yogurt),(Kidney Beans),0.6,1.0,0.6,1.00,1.00,1.0,0.00,inf,0.0,0.60,0.000,0.800
3,(Eggs),(Onion),0.8,0.6,0.6,0.75,1.25,1.0,0.12,1.6,1.0,0.75,0.375,0.875
4,(Onion),(Eggs),0.6,0.8,0.6,1.00,1.25,1.0,0.12,inf,0.5,0.75,1.000,0.875
5,(Onion),(Kidney Beans),0.6,1.0,0.6,1.00,1.00,1.0,0.00,inf,0.0,0.60,0.000,0.800
6,"(Eggs, Kidney Beans)",(Onion),0.8,0.6,0.6,0.75,1.25,1.0,0.12,1.6,1.0,0.75,0.375,0.875
7,"(Eggs, Onion)",(Kidney Beans),0.6,1.0,0.6,1.00,1.00,1.0,0.00,inf,0.0,0.60,0.000,0.800
8,"(Onion, Kidney Beans)",(Eggs),0.6,0.8,0.6,1.00,1.25,1.0,0.12,inf,0.5,0.75,1.000,0.875
9,(Eggs),"(Onion, Kidney Beans)",0.8,0.6,0.6,0.75,1.25,1.0,0.12,1.6,1.0,0.75,0.375,0.875


## Example 2 -- Rule Generation and Selection Criteria

If you are interested in rules according to a different metric of interest, you can simply adjust the metric and min_threshold arguments . E.g. if you are only interested in rules that have a lift score of >= 1.2, you would do the following:

In [15]:
rules = association_rules(frequent_itemsets, metric="lift", min_threshold=1.2)
rules

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
0,(Eggs),(Onion),0.8,0.6,0.6,0.75,1.25,1.0,0.12,1.6,1.0,0.75,0.375,0.875
1,(Onion),(Eggs),0.6,0.8,0.6,1.00,1.25,1.0,0.12,inf,0.5,0.75,1.000,0.875
2,"(Eggs, Kidney Beans)",(Onion),0.8,0.6,0.6,0.75,1.25,1.0,0.12,1.6,1.0,0.75,0.375,0.875
3,"(Onion, Kidney Beans)",(Eggs),0.6,0.8,0.6,1.00,1.25,1.0,0.12,inf,0.5,0.75,1.000,0.875
4,(Eggs),"(Onion, Kidney Beans)",0.8,0.6,0.6,0.75,1.25,1.0,0.12,1.6,1.0,0.75,0.375,0.875
5,(Onion),"(Eggs, Kidney Beans)",0.6,0.8,0.6,1.00,1.25,1.0,0.12,inf,0.5,0.75,1.000,0.875


Pandas DataFrames make it easy to filter the results further. Let's say we are ony interested in rules that satisfy the following criteria:

1. at least 2 antecedents
2. a confidence > 0.75
3. a lift score > 1.2
We could compute the antecedent length as follows:

In [23]:
rules["antecedent_len"] = rules["antecedents"].apply(lambda x: len(x))
rules

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski,antecedent_len
0,(Eggs),(Onion),0.8,0.6,0.6,0.75,1.25,1.0,0.12,1.6,1.0,0.75,0.375,0.875,1
1,(Onion),(Eggs),0.6,0.8,0.6,1.00,1.25,1.0,0.12,inf,0.5,0.75,1.000,0.875,1
2,"(Eggs, Kidney Beans)",(Onion),0.8,0.6,0.6,0.75,1.25,1.0,0.12,1.6,1.0,0.75,0.375,0.875,2
3,"(Onion, Kidney Beans)",(Eggs),0.6,0.8,0.6,1.00,1.25,1.0,0.12,inf,0.5,0.75,1.000,0.875,2
4,(Eggs),"(Onion, Kidney Beans)",0.8,0.6,0.6,0.75,1.25,1.0,0.12,1.6,1.0,0.75,0.375,0.875,1
5,(Onion),"(Eggs, Kidney Beans)",0.6,0.8,0.6,1.00,1.25,1.0,0.12,inf,0.5,0.75,1.000,0.875,1


Then, we can use pandas' selection syntax as shown below:


In [25]:
# at least 2 antecedents &  confidence > 0.75 &  a lift score > 1.2
rules[(rules['antecedent_len'] >= 2) & (rules['confidence'] > 0.75) &  (rules['lift'] > 1.2)]  

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski,antecedent_len
3,"(Onion, Kidney Beans)",(Eggs),0.6,0.8,0.6,1.0,1.25,1.0,0.12,inf,0.5,0.75,1.0,0.875,2


Similarly, using the Pandas API, we can select entries based on the "antecedents" or "consequents" columns:

In [26]:
rules[rules['antecedents'] == {'Eggs', 'Kidney Beans'}]


,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski,antecedent_len
2,"(Eggs, Kidney Beans)",(Onion),0.8,0.6,0.6,0.75,1.25,1.0,0.12,1.6,1.0,0.75,0.375,0.875,2


**Frozensets**

Note that the entries in the "itemsets" column are of type **frozenset**, which is built-in Python type that is **similar to a Python set but immutable**, which makes it more efficient for certain query or comparison operations (https://docs.python.org/3.6/library/stdtypes.html#frozenset). Since frozensets are sets, the **item order does not matter**. I.e., the query

* rules[rules['antecedents'] == {'Eggs', 'Kidney Beans'}]

is equivalent to any of the following three

* rules[rules['antecedents'] == {'Kidney Beans', 'Eggs'}]
* rules[rules['antecedents'] == frozenset(('Eggs', 'Kidney Beans'))]
* rules[rules['antecedents'] == frozenset(('Kidney Beans', 'Eggs'))]

## Example 3 -- Frequent Itemsets with Incomplete Antecedent and Consequent Information

**Most metrics computed by association_rules depends*** on the **consequent** and **antecedent** ```support score``` of a **given rule** provided in the **frequent itemset input DataFrame**. Consider the following example:

In [29]:
import pandas as pd

dict = {'itemsets': [['177', '176'], ['177', '179'],
                     ['176', '178'], ['176', '179'],
                     ['93', '100'], ['177', '178'],
                     ['177', '176', '178']],
        'support':[0.253623, 0.253623, 0.217391,
                   0.217391, 0.181159, 0.108696, 0.108696]}

freq_itemsets = pd.DataFrame(dict)
freq_itemsets

,itemsets,support
0,"[177, 176]",0.253623
1,"[177, 179]",0.253623
2,"[176, 178]",0.217391
3,"[176, 179]",0.217391
4,"[93, 100]",0.181159
5,"[177, 178]",0.108696
6,"[177, 176, 178]",0.108696


> Note that this is a "cropped" DataFrame that **doesn't contain the support values of the item subsets**. This can create problems if we want to compute the association rule metrics for, e.g.,**176 => 177**.

In these scenarios, **where not all metric's can be computed**, due to incomplete input DataFrames, you can use the **support_only=True** option, which **will only compute the support column of a given rule that does not require as much info**:

$support(A→C)=support(A∪C),range: [0,1]$

"NaN's" will be assigned to all other metric columns:

In [30]:
from mlxtend.frequent_patterns import association_rules

res = association_rules(freq_itemsets, support_only=True, min_threshold=0.1)
res

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
0,(177),(176),NaN,NaN,0.253623,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,(176),(177),NaN,NaN,0.253623,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,(177),(179),NaN,NaN,0.253623,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,(179),(177),NaN,NaN,0.253623,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,(176),(178),NaN,NaN,0.217391,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,(178),(176),NaN,NaN,0.217391,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,(179),(176),NaN,NaN,0.217391,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,(176),(179),NaN,NaN,0.217391,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,(93),(100),NaN,NaN,0.181159,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,(100),(93),NaN,NaN,0.181159,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


To clean up the representation, you may want to do the following:

In [31]:
res = res[['antecedents', 'consequents', 'support']]
res

,antecedents,consequents,support
0,(177),(176),0.253623
1,(176),(177),0.253623
2,(177),(179),0.253623
3,(179),(177),0.253623
4,(176),(178),0.217391
5,(178),(176),0.217391
6,(179),(176),0.217391
7,(176),(179),0.217391
8,(93),(100),0.181159
9,(100),(93),0.181159


## Example 4 -- Pruning Association Rules


There is no specific API for pruning. Instead, the pandas API can be used on the resulting data frame to remove individual rows. E.g., suppose we have the following rules:

In [32]:
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, fpmax, fpgrowth
from mlxtend.frequent_patterns import association_rules


dataset = [['Milk', 'Onion', 'Nutmeg', 'Kidney Beans', 'Eggs', 'Yogurt'],
           ['Dill', 'Onion', 'Nutmeg', 'Kidney Beans', 'Eggs', 'Yogurt'],
           ['Milk', 'Apple', 'Kidney Beans', 'Eggs'],
           ['Milk', 'Unicorn', 'Corn', 'Kidney Beans', 'Yogurt'],
           ['Corn', 'Onion', 'Onion', 'Kidney Beans', 'Ice cream', 'Eggs']]

te = TransactionEncoder()
te_ary = te.fit(dataset).transform(dataset)
df = pd.DataFrame(te_ary, columns=te.columns_)

frequent_itemsets = fpgrowth(df, min_support=0.6, use_colnames=True)
rules = association_rules(frequent_itemsets, metric="lift", min_threshold=1.2)
rules

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
0,(Eggs),(Onion),0.8,0.6,0.6,0.75,1.25,1.0,0.12,1.6,1.0,0.75,0.375,0.875
1,(Onion),(Eggs),0.6,0.8,0.6,1.00,1.25,1.0,0.12,inf,0.5,0.75,1.000,0.875
2,"(Eggs, Kidney Beans)",(Onion),0.8,0.6,0.6,0.75,1.25,1.0,0.12,1.6,1.0,0.75,0.375,0.875
3,"(Onion, Kidney Beans)",(Eggs),0.6,0.8,0.6,1.00,1.25,1.0,0.12,inf,0.5,0.75,1.000,0.875
4,(Eggs),"(Onion, Kidney Beans)",0.8,0.6,0.6,0.75,1.25,1.0,0.12,1.6,1.0,0.75,0.375,0.875
5,(Onion),"(Eggs, Kidney Beans)",0.6,0.8,0.6,1.00,1.25,1.0,0.12,inf,0.5,0.75,1.000,0.875


and we **want to remove the rule "(Onion, Kidney Beans) -> (Eggs)"**. In order to to this, we can define selection masks and remove this row as follows:

In [35]:
antecedent_sele = rules['antecedents'] == frozenset({'Onion', 'Kidney Beans'}) # or  frozenset({'Kidney Beans', 'Onion'})
consequent_sele = rules['consequents'] == frozenset({'Eggs'})
final_sele = (antecedent_sele & consequent_sele)

rules.loc[ ~final_sele ]

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
0,(Eggs),(Onion),0.8,0.6,0.6,0.75,1.25,1.0,0.12,1.6,1.0,0.75,0.375,0.875
1,(Onion),(Eggs),0.6,0.8,0.6,1.00,1.25,1.0,0.12,inf,0.5,0.75,1.000,0.875
2,"(Eggs, Kidney Beans)",(Onion),0.8,0.6,0.6,0.75,1.25,1.0,0.12,1.6,1.0,0.75,0.375,0.875
4,(Eggs),"(Onion, Kidney Beans)",0.8,0.6,0.6,0.75,1.25,1.0,0.12,1.6,1.0,0.75,0.375,0.875
5,(Onion),"(Eggs, Kidney Beans)",0.6,0.8,0.6,1.00,1.25,1.0,0.12,inf,0.5,0.75,1.000,0.875
